In [11]:
import numpy as np
from scipy import stats

The goal of this code is to produce a HMM on which the viterbi algorithm can be run.

In [2]:
def EmissionProbs(observation, hidden_state):
    # This function needs to output P(observation|Hidden state)
    return 0.5 * observation * (0.9 - hidden_state)

In [3]:
class HMM:
    def __init__(self, n_components, transmat, startprob, EmissionProbs):
        self.N = n_components

        # These define the HMM
        self.transmat = transmat
        # The columns of the transition matrix need to be in the same order as the components in transmat
        # The rows of the transition matrix needs to be the opposite order of the hidden state vector

        self.pi = startprob
        self.EmissionProbs = EmissionProbs # This is a function

    def decode(self, observ):
        # observ is the observation sequence that is to be decoded to get the hidden state sequence

        viterbi = [[0] * self.N] * len(observ)
        backpointer = [0] * len(observ)

        for s in range(self.N):
            viterbi[0][s] = self.pi[s] * self.EmissionProbs(observ[0], s)
            backpointer[0] = viterbi[0].index(max(viterbi[0]))

        for t in range(1, len(observ)):
            viterbi[t] = list(np.matmul( self.transmat, viterbi[t-1] ) * np.array([self.EmissionProbs(observ[0], i) for i in range(self.N)]))

            backpointer[t] = viterbi[t].index(max(viterbi[t]))

        return backpointer


In [4]:
model = HMM(2, [[0.1, 0.9], [0.3, 0.7]], [0.2, 0.8], EmissionProbs)

In [5]:
model.decode([3,5,5,2,3])

[0, 1, 1, 0, 0]

GMM hidden layer

In [ ]:
# The goal of this segment is to "train" the GMM such that we can find the mean vector and the covariance matrix required to model the data

In [19]:
# For all the feature vectors of a particular character, find their mean vector and the covariance matrix

def mean_vector(X): # The input is going to be a 2D matrix (each row is a new case)
    return np.mean(X, axis=0)

def Cov_matrix(X):
    return np.cov(X, rowvar=False)

In [97]:
X = [[2,6,3,3,2], [2,6,3,3,2], [3,6,3,2,3]]
mu = mean_vector(X)
cov = Cov_matrix(X)

In [96]:
X_test = np.random.multivariate_normal(mu, cov)
stats.multivariate_normal.pdf(X_test, mu, cov, allow_singular=True)

0.33640962460077906

In [142]:
class letter:
    def __init__(self, X): # X is the set of observations that correspond to a character (2D list)
        self.mu = mean_vector(X)
        self.cov = Cov_matrix(X)

        # The below is for the purpose of covariance update
        def multiply_with_transpose(arrays):
            res = np.array([[0] * len(X[1])] * len(X[1]))
            for array in arrays:
                array = np.array([array])
                res += np.matmul(np.transpose(array), array)
            return np.array(res)
        
        self.A = multiply_with_transpose(X)

        self.n = len(X)

    def GetProb(self, input):
        return stats.multivariate_normal.pdf(input, self.mu, self.cov, allow_singular=True)
    
    def Update(self, X): # 1D vector corresponding to 1 new case
        def Update_mean(X, Xnm1, n): # The input is one case - one 1D vector; Xnm1 - X sub(n - 1);n number of instance gone into n
            return 1 / (n+1) * ( X + (n) * Xnm1 )
        
        self.mu = Update_mean(X, self.mu, self.n)

        self.b = self.mu * (self.n+1)

        def UpdateCov(A, B, mu, n):
            t1 = np.matmul(mu, np.transpose(B))
            t2 = np.matmul(np.transpose(mu), B)
            t3 = np.matmul(mu, np.transpose(mu))
            return 1 / (n+1) * ( A - t1 - t2 + n * t3)
        
        self.cov = UpdateCov(self.A, self.b, self.mu, self.n)

        self.n += 1



In [143]:
l = letter([[1,4,5],[3,5,8], [5,7,8], [5,0,8]])
print(l.cov)
l.Update([4,1,9])
print(l.cov)

# Fix this broken covariance update

[[ 3.66666667 -0.66666667  2.5       ]
 [-0.66666667  8.66666667  0.        ]
 [ 2.5         0.          2.25      ]]
[[-86.736 -87.936 -76.936]
 [-87.936 -80.736 -75.536]
 [-76.936 -75.536 -55.336]]


CNN autoencoder

In [13]:
# The goal of this section is to convert image into feature vector

x_train = [None] # This is supposed to be the 

image_height = 64
image_width = 64

# x_train = np.reshape(x_train, (len(x_train), image_width, image_height, 1)) 

In [22]:
from tensorflow.keras import layers
from keras.layers import Input, Dense, Conv2D, MaxPooling2D, UpSampling2D, Flatten
from keras.models import Model
from keras import backend as K
import tensorflow as tf

In [32]:
input_img = Input(shape=(image_width, image_height, 1)) # Adjust channels as needed

x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
encoded = Dense(8, activation='relu')(x)
# This is the layer that is going to later become the feature vector: shape (None, 16, 16, 32) 

x = Dense(8, activation='relu')(encoded)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
decoded = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

autoencoder = Model(input_img, decoded)
autoencoder.summary()

Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_25 (InputLayer)       [(None, 64, 64, 1)]       0         
                                                                 
 conv2d_81 (Conv2D)          (None, 64, 64, 32)        320       
                                                                 
 max_pooling2d_36 (MaxPooli  (None, 32, 32, 32)        0         
 ng2D)                                                           
                                                                 
 conv2d_82 (Conv2D)          (None, 32, 32, 32)        9248      
                                                                 
 max_pooling2d_37 (MaxPooli  (None, 16, 16, 32)        0         
 ng2D)                                                           
                                                                 
 dense_4 (Dense)             (None, 16, 16, 8)         264 

In [34]:
class Autoencoder(Model):
  def __init__(self, image_width, image_height):
    super(Autoencoder, self).__init__()

    self.encoder = tf.keras.Sequential([
      Input(shape=(image_width, image_height, 1)),
      Conv2D(32, (3, 3), activation='relu', padding='same'),
      MaxPooling2D((2, 2), padding='same'),
      Conv2D(32, (3, 3), activation='relu', padding='same'),
      MaxPooling2D((2, 2), padding='same'),
      Dense(8, activation='relu')
    ])
    self.decoder = tf.keras.Sequential([
      Dense(8, activation='relu'),
      Conv2D(32, (3, 3), activation='relu', padding='same'),
      UpSampling2D((2, 2)),
      Conv2D(32, (3, 3), activation='relu', padding='same'),
      UpSampling2D((2, 2)),
      Conv2D(1, (3, 3), activation='sigmoid', padding='same')
    ])

  def call(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded


autoencoder = Autoencoder(image_height, image_width)
image = tf.random.uniform((image_height, image_width, 1), minval=0, maxval=1, dtype=tf.float32)
autoencoder(image)
autoencoder.summary()

Model: "autoencoder_15"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_27 (Sequential)  (None, 16, 16, 32)        9568      
                                                                 
 sequential_28 (Sequential)  (64, 64, 4, 1)            18785     
                                                                 
Total params: 28353 (110.75 KB)
Trainable params: 28353 (110.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder.fit(x_train, x_train, # X = Y here
                epochs=20,
                batch_size=256,
                shuffle=True)

In [ ]:
encoded_imgs = autoencoder.encoder(X_test).numpy()
decoded_imgs = autoencoder.decoder(encoded_imgs).numpy()